# 01 - Refusal direction on Qwen2.5-1.5B-Instruct (local GPU)

Replicates the core result of Arditi et al. (2024), *Refusal in Language Models Is Mediated by a
Single Direction* ([arXiv:2406.11717](https://arxiv.org/abs/2406.11717)), on a model small
enough for a 6 GB GPU in fp16.

All logic lives in `src/refusal_direction`; this notebook only configures and calls it:

1. Load 128 harmful (AdvBench) + 128 harmless (Alpaca) training instructions and 32 + 32 held-out validation instructions, with pinned sources and SHA-256 checks.
2. Capture the residual stream at the last prompt token after every layer.
3. Compute one difference-in-means direction per layer.
4. For every layer: ablate its direction from every layer's residual stream and measure the refusal rate on harmful validation prompts; add it at that layer and measure the refusal rate on harmless validation prompts.
5. Save `refusal_by_layer.csv`, `refusal_by_layer.png`, `summary.json` and the completions in `results/qwen2.5-1.5b-instruct/`.

Refusal is scored by substring match (the paper's phrase list). Runtime on an RTX 2060: see `summary.json`.

In [ ]:
import os, sys, json, time
from pathlib import Path

import torch

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (REPO / "pyproject.toml").exists(), "run from the repo root or notebooks/"
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from refusal_direction import __version__
from refusal_direction.data import get_splits
from refusal_direction.model import load_model, format_prompts
from refusal_direction.pipeline import ExperimentConfig, run_experiment
from refusal_direction.scoring import is_refusal, REFUSAL_SUBSTRINGS

print("refusal_direction", __version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
cfg = ExperimentConfig(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    out_dir=REPO / "results" / "qwen2.5-1.5b-instruct",
    dtype="float16",          # RTX 2060 (Turing) has no bf16; Qwen2.5 is stable in fp16
    n_train=128, n_val=32,    # as in the paper
    seed=0,
    position=-1,              # last token of the chat-templated prompt
    max_new_tokens=64,
    batch_size=16,
    splits_path=REPO / "data" / "splits_seed0.json",
)
cfg

In [ ]:
splits = get_splits(cfg.splits_path, cfg.n_train, cfg.n_val, cfg.seed)
for k in ("harmful_train", "harmful_val", "harmless_train", "harmless_val"):
    print(f"{k:15s} {len(splits[k])}")
print("\nsources:")
for name, src in splits["sources"].items():
    print(f"  {name}: {src['url']}\n    sha256 {src['sha256']}")
print("\nexample harmless train prompts:")
for p in splits["harmless_train"][:3]:
    print("  -", p)

In [ ]:
bundle = load_model(cfg.model_name, dtype=getattr(torch, cfg.dtype))
print(f"{bundle.name}: {bundle.n_layers} layers, d_model={bundle.d_model}, {bundle.dtype}, {bundle.device}")
print("GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))
print("\nchat-templated prompt (the direction is read at its last token):")
print(repr(format_prompts(bundle, [splits["harmless_val"][0]])[0]))

## Run the experiment

Prints one line per layer: refusal rate on harmful validation prompts with that layer's direction
ablated everywhere, and refusal rate on harmless validation prompts with that layer's raw
difference vector added at that layer.

In [ ]:
res = run_experiment(cfg, bundle=bundle)
summary = res["summary"]
print(json.dumps({k: v for k, v in summary.items() if k != "config"}, indent=1))

In [ ]:
df = res["df"]
df.style.format({c: "{:.3f}" for c in df.columns if "rate" in c or c == "direction_norm"}).hide(axis="index")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(cfg.out_dir / "refusal_by_layer.png")))

## Examples at the chosen layer

Harmless prompts with the direction **added** (benign prompts, so the induced refusals are shown in
full). For harmful prompts with the direction **ablated**, only the per-prompt refusal flag is shown:
the completions are saved locally in `completions_harmful_ablation.jsonl` but are not committed to
the public repo.

In [ ]:
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")]

add = load_jsonl(cfg.out_dir / "completions_harmless_addition.jsonl")
base = load_jsonl(cfg.out_dir / "completions_harmless_baseline.jsonl")
for b, a in list(zip(base, add))[:4]:
    print("PROMPT:   ", a["instruction"])
    print("baseline: ", b["completion"][:160].replace("\n", " "), "| refusal =", is_refusal(b["completion"]))
    print("+direction:", a["completion"][:160].replace("\n", " "), "| refusal =", is_refusal(a["completion"]))
    print()

In [ ]:
abl = load_jsonl(cfg.out_dir / "completions_harmful_ablation.jsonl")
hb = load_jsonl(cfg.out_dir / "completions_harmful_baseline.jsonl")
print(f"harmful val: baseline refusals {sum(is_refusal(x['completion']) for x in hb)}/{len(hb)}, "
      f"ablated refusals {sum(is_refusal(x['completion']) for x in abl)}/{len(abl)}")
print("per-prompt refusal flags after ablation:", "".join("R" if is_refusal(x["completion"]) else "." for x in abl))